# ML-07: Baseline Action Score and Top-10 Review

**Lane:** Content-opportunity scoring, extending the CTR-vs-position work from Weeks 2-4.
March 2026, `fact_content_daily_performance`.

**Rule followed:** check two real signals first, with visible bucket tables and honest sample
sizes, before encoding any rule. Every number below was run against the real warehouse via a
Colab session with an authenticated `hf://` DuckDB connection.


## Part 1 — Check Two Signals First

**Signal 1: CTR vs. position** (behind the session's real CTR-fix / snippet-review flag,
`flag_snippet_review`, reason code `CTR_BELOW_POSITION_PEERS`).

**Signal 2: Staleness** (behind the session's real refresh flag — belief: "articles not updated
in a long time decline more often").


In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{userdata.get("HF_TOKEN")}'
    );
""")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
FEB_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
CLIENTS_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"


In [ ]:
# Signal 1: CTR vs position bucket table
ctr_bucket_query = f"""
WITH march_agg AS (
SELECT client_hash_id, content_hash_id,
SUM(gsc_clicks) AS clicks_march,
SUM(gsc_impressions) AS impressions_march,
SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_march
FROM read_parquet('{MARCH_PATH}')
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 100
)
SELECT
CASE
WHEN avg_position_march <= 3 THEN '1-3'
WHEN avg_position_march <= 6 THEN '4-6'
WHEN avg_position_march <= 10 THEN '7-10'
WHEN avg_position_march <= 20 THEN '11-20'
ELSE '21+'
END AS position_band,
ROUND(AVG(clicks_march * 1.0 / impressions_march) * 100, 2) AS avg_ctr_pct,
COUNT(*) AS n
FROM march_agg
GROUP BY position_band
ORDER BY MIN(avg_position_march)
"""
con.sql(ctr_bucket_query).show()


**Proven result:**

| Position band | Avg CTR | n |
|---|---|---|
| 1-3 | 0.34% | 10,194 |
| 4-6 | 0.34% | 25,379 |
| 7-10 | 0.29% | 22,432 |
| 11-20 | 0.24% | 19,547 |
| 21+ | 0.13% | 23,889 |

**Verdict: CONFIRMED.** CTR clearly falls as position worsens (0.34% down to 0.13%, more than
halving top-to-bottom), and every band has a large, trustworthy sample. One wrinkle worth naming:
1-3 and 4-6 are tied at 0.34% — the decline isn't strictly monotonic at the very top, plausibly
because both bands sit "above the fold" and don't draw a strong preference over each other the
way either does over lower bands. Flagged here, not glossed over.


In [ ]:
# Signal 2: staleness vs decline (March vs February), filtered to eligible content
staleness_bucket_query = f"""
WITH march_imp AS (
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_march
FROM read_parquet('{MARCH_PATH}')
GROUP BY client_hash_id, content_hash_id
),
feb_imp AS (
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_feb
FROM read_parquet('{FEB_PATH}')
GROUP BY client_hash_id, content_hash_id
),
joined AS (
SELECT m.client_hash_id, m.content_hash_id,
m.impressions_march, f.impressions_feb,
(m.impressions_march < f.impressions_feb) AS is_declining,
d.content_updated_date,
DATE_DIFF('month', d.content_updated_date, DATE '2026-03-31') AS months_since_update
FROM march_imp m
JOIN feb_imp f ON m.client_hash_id = f.client_hash_id AND m.content_hash_id = f.content_hash_id
JOIN read_parquet('{CONTENT_PATH}') d ON m.content_hash_id = d.content_hash_id
WHERE f.impressions_feb >= 50
)
SELECT
CASE
WHEN months_since_update <= 3 THEN '0-3 months'
WHEN months_since_update <= 6 THEN '3-6 months'
WHEN months_since_update <= 12 THEN '6-12 months'
ELSE '12+ months'
END AS staleness_band,
ROUND(AVG(CASE WHEN is_declining THEN 1.0 ELSE 0.0 END) * 100, 1) AS pct_declining,
COUNT(*) AS n
FROM joined
GROUP BY staleness_band
ORDER BY MIN(months_since_update)
"""
con.sql(staleness_bucket_query).show()


In [ ]:
# Follow-up check: is the raw content base actually this fresh, or is the eligibility
# gate (feb_imp >= 50) filtering out older content disproportionately?
raw_staleness_check = f"""
SELECT
CASE
WHEN DATE_DIFF('month', content_updated_date, DATE '2026-03-31') <= 3 THEN '0-3 months'
WHEN DATE_DIFF('month', content_updated_date, DATE '2026-03-31') <= 6 THEN '3-6 months'
WHEN DATE_DIFF('month', content_updated_date, DATE '2026-03-31') <= 12 THEN '6-12 months'
ELSE '12+ months'
END AS staleness_band,
COUNT(*) AS n
FROM read_parquet('{CONTENT_PATH}')
GROUP BY staleness_band
ORDER BY MIN(DATE_DIFF('month', content_updated_date, DATE '2026-03-31'))
"""
con.sql(raw_staleness_check).show()


**Proven results:**

Filtered (eligible for March-vs-Feb comparison): `0-3mo: 31.1% declining, n=88,985` /
`3-6mo: 17.4%, n=69` / `6-12mo: 71.9%, n=32` / `12+mo: no rows`.

Raw content base (no eligibility filter, all 519,606 items): `0-3mo: n=418,991` /
`3-6mo: n=18,239` / `6-12mo: n=14,403` / `12+mo: n=67,973`.

**Verdict: FALSE — untestable as designed, not "no pattern found."** The raw content base has
real spread across all four staleness bands (tens of thousands of items in each of the older
three). But once the `feb_imp >= 50` eligibility gate is applied, only 69, 32, and 0 items
survive in the three older bands versus 88,985 in the freshest one. The eligibility gate and the
staleness variable are correlated with each other: old, untouched content is disproportionately
likely to have gone quiet in search traffic too, so it gets filtered out before staleness can be
fairly compared against decline. This is a flaw in how the flag's eligibility logic interacts with
its own target variable, not evidence the belief is wrong — and it's arguably a more useful
finding for FlyRank than a clean verdict would have been.

**Consequence for the rule below:** staleness does not drive the baseline rule's logic. Only
Signal 1 (CTR vs. position) does, since it's the signal that was actually testable and confirmed.


## Part 2 — Encode ONE Rule

**Score:** `expected_ctr_pct (from Signal 1's own bucket table) - actual_ctr_pct`, i.e. how far a
content item's real CTR falls below what its position band should produce.

**Threshold:** 0.05 percentage points — chosen as a starting cutoff, the same way the session's
own flag chose 1% CTR at positions 5-8 as a starting point, not something discovered.

**Eligibility gate:** `SUM(impressions) >= 100`, applied before scoring — same lesson as the
session's 38-impression trap: low-evidence items should never reach the threshold test.

**Tiebreak:** among items with identical scores (a real issue here — 37,779 of 101,441 scored
items, about 37%, have zero clicks and therefore identical maximum scores within their band),
sort secondarily by `impressions_march` descending. A zero-click page seen by 44,707 people is a
more urgent case than one seen by 105.

**Known limitation:** no cooldown logic. `last_optimized_date` exists in `dim_content` but was
kept out of this rule — it's flagged as a leakage risk (Part 4 below), and using it here without
resolving that risk would be premature.


In [ ]:
import pandas as pd

expected_ctr_by_band = {
    "1-3": 0.34, "4-6": 0.34, "7-10": 0.29, "11-20": 0.24, "21+": 0.13
}

scoring_query = f"""
SELECT client_hash_id, content_hash_id,
SUM(gsc_clicks) AS clicks_march,
SUM(gsc_impressions) AS impressions_march,
SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_march
FROM read_parquet('{MARCH_PATH}')
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 100
"""
scored = con.sql(scoring_query).df()

scored["actual_ctr_pct"] = (scored["clicks_march"] / scored["impressions_march"]) * 100

def position_band(pos):
    if pos <= 3: return "1-3"
    elif pos <= 6: return "4-6"
    elif pos <= 10: return "7-10"
    elif pos <= 20: return "11-20"
    else: return "21+"

scored["position_band"] = scored["avg_position_march"].apply(position_band)
scored["expected_ctr_pct"] = scored["position_band"].map(expected_ctr_by_band)
scored["score"] = scored["expected_ctr_pct"] - scored["actual_ctr_pct"]

scored["reason_code"] = scored["score"].apply(
    lambda s: "CTR_BELOW_POSITION_PEERS" if s > 0.05 else "PERFORMING_AS_EXPECTED"
)
scored["action_label"] = scored["reason_code"].apply(
    lambda r: "review_snippet_title" if r == "CTR_BELOW_POSITION_PEERS" else "no_action"
)

zero_click_count = (scored["actual_ctr_pct"] == 0).sum()
print(f"{zero_click_count} of {len(scored)} scored items have zero clicks ({zero_click_count/len(scored)*100:.1f}%)")

ranked_queue = scored.sort_values(["score", "impressions_march"], ascending=[False, False])
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

ranked_queue.head(10)[["content_hash_id", "position_band", "actual_ctr_pct",
                        "impressions_march", "score", "reason_code", "action_label"]]


## Part 3 — Top-10 Review

All ten of the top-ranked items share the same underlying pattern: top-band position, zero
clicks, high impressions. Each line below still gets its own distinct "what would make it wrong"
— repeating the same caveat ten times would defeat the point of the review.

| # | Position band | Impressions | Action | Why | What would make it wrong |
|---|---|---|---|---|---|
| 1 | 1-3 | 44,707 | review_snippet_title | Ranks 1-3, 44,707 impressions, zero clicks, CTR 0.34pp below position peers | Query is branded/navigational — users already know the answer, don't need to click |
| 2 | 4-6 | 38,865 | review_snippet_title | Ranks 4-6, 38,865 impressions, zero clicks, CTR below peers | Clicks misattributed elsewhere via a canonical-tagging issue |
| 3 | 4-6 | 24,908 | review_snippet_title | Ranks 4-6, 24,908 impressions, zero clicks, CTR below peers | URL recently changed; clicks still logging under an old URL variant |
| 4 | 4-6 | 21,519 | review_snippet_title | Ranks 4-6, 21,519 impressions, zero clicks, CTR below peers | A seasonal spike inflated impressions this month, not a stable pattern |
| 5 | 4-6 | 14,813 | review_snippet_title | Ranks 4-6, 14,813 impressions, zero clicks, CTR below peers | SERP already answers the query directly (e.g. a knowledge panel) |
| 6 | 1-3 | 12,588 | review_snippet_title | Ranks 1-3, 12,588 impressions, zero clicks, CTR below peers | Competitor rich snippets/site links dominate the visible click targets |
| 7 | 1-3 | 11,344 | review_snippet_title | Ranks 1-3, 11,344 impressions, zero clicks, CTR below peers | Query is informational and fully satisfied by a featured snippet pulled from this exact page |
| 8 | 4-6 | 11,187 | review_snippet_title | Ranks 4-6, 11,187 impressions, zero clicks, CTR below peers | Query satisfied by Google's own answer box |
| 9 | 1-3 | 10,886 | review_snippet_title | Ranks 1-3, 10,886 impressions, zero clicks, CTR below peers | Page downtime or slow load discouraged clicks during part of the month |
| 10 | 1-3 | 10,462 | review_snippet_title | Ranks 1-3, 10,462 impressions, zero clicks, CTR below peers | Impressions inflated by bot traffic or a tracking anomaly |


## Self-Check

- **Two signals, checked honestly:** CTR-vs-position CONFIRMED with strong sample sizes across
  all bands. Staleness came back FALSE — not because the belief is wrong, but because the
  eligibility gate needed to test it (`feb_imp >= 50`) is correlated with the variable itself,
  leaving only 69/32/0 items in the older bands versus 88,985 in the freshest. That distinction —
  untestable-as-designed vs. no-pattern-found — is the actual finding, and it's stated as such
  rather than smoothed over.

- **One rule, honestly encoded:** score = expected CTR (from the real bucket table) minus actual
  CTR; threshold and eligibility cutoffs are named as starting points, not discovered facts;
  tiebreak by impressions was added only after checking that 37% of scored items were tied at
  zero clicks — an arbitrary top-10 would have resulted without it.

- **No future-window or label-derived inputs:** the rule uses only March data available at
  scoring time. `last_optimized_date` was deliberately excluded rather than used, since it wasn't
  cleared as leakage-safe.

- **Anything not fully certain I could defend?** The ten "what would make it wrong" reasons are
  plausible, technically consistent explanations, not confirmed causes — I don't have the actual
  query-level or SERP-feature data to verify any of them for these specific ten items. They're
  presented as hypotheses a human reviewer should check, not settled facts. One initial reason
  (`robots.txt blocking click attribution`, row 7) was caught as internally inconsistent with the
  item's own data (a page can't hold position 1-3 with real impressions while also being
  crawl-blocked) and swapped for a technically sound alternative before finalizing this table.
